# Machine Learning Process
Load the dataset, train models, and save artifacts.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
import shap

FEATURES = [
    'tenure_months', 'total_orders', 'total_spend', 'recency_days', 'age', 'income_bracket', 
    'nps_score', 'online_ratio', 'return_rate', 'support_tickets', 'discount_usage', 'avg_order_value', 
    'tenure_years', 'purchase_frequency', 'recency_score', 'frequency_score', 'monetary_score', 
    'rfm_score', 'retention_rate', 'churn_rate'
]
TARGET = 'clv_12month'

df = pd.read_csv("../data/customers.csv")
X = df[FEATURES].copy()
y = df[TARGET].copy()

y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")

y_test_dollars = np.expm1(y_test)

ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
joblib.dump(ridge, "../models/ridge_model.pkl")

rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_split=10, 
                           min_samples_leaf=5, max_features='sqrt', n_jobs=-1, random_state=42)
rf.fit(X_train_scaled, y_train)
joblib.dump(rf, "../models/rf_model.pkl")

xgb = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, 
                   colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, random_state=42, eval_metric='rmse')
xgb.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)
joblib.dump(xgb, "../models/xgb_model.pkl")

y_pred_rf_log = rf.predict(X_test_scaled)
y_pred_xgb_log = xgb.predict(X_test_scaled)
y_pred_ridge_log = ridge.predict(X_test_scaled)

explainer = shap.TreeExplainer(rf)
joblib.dump(explainer, "../models/shap_explainer.pkl")

X_all_scaled = scaler.transform(df[FEATURES])
df['clv_predicted_rf'] = np.expm1(rf.predict(X_all_scaled))
df['clv_predicted_ridge'] = np.expm1(ridge.predict(X_all_scaled))
df['clv_predicted_xgb'] = np.expm1(xgb.predict(X_all_scaled))
df['clv_predicted_ensemble'] = np.expm1(0.5 * rf.predict(X_all_scaled) + 0.3 * xgb.predict(X_all_scaled) + 0.2 * ridge.predict(X_all_scaled))
df['clv_error_rf'] = np.abs(df['clv_12month'] - df['clv_predicted_rf'])

df.to_csv("../data/customers_clv.csv", index=False)

def evaluate_model(y_true, y_pred, name):
    return {'MAE': mean_absolute_error(y_true, y_pred), 'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)), 
            'R2': r2_score(y_true, y_pred), 'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100}

metrics_all = {
    'Ridge': evaluate_model(y_test_dollars, np.expm1(y_pred_ridge_log), "Ridge"),
    'Random Forest': evaluate_model(y_test_dollars, np.expm1(y_pred_rf_log), "Random Forest"),
    'XGBoost': evaluate_model(y_test_dollars, np.expm1(y_pred_xgb_log), "XGBoost"),
    'Ensemble': evaluate_model(y_test_dollars, np.expm1(0.5 * y_pred_rf_log + 0.3 * y_pred_xgb_log + 0.2 * y_pred_ridge_log), "Ensemble")
}
joblib.dump(metrics_all, "../models/metrics.pkl")
print("ML Training Complete!")


## Review Models Performance

In [ ]:
for model, m in metrics_all.items():
    print(f"{model}: R2={m['R2']:.4f}, MAE=${m['MAE']:.2f}")